In [ ]:
#Classical Method

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten, LeakyReLU, ELU
from tensorflow.keras.optimizers import SGD

df = pd.read_excel("Final Data Compiled for BeQu.xlsx", sheet_name='FinalData')

confirmed = df[df['koi_disposition'] == 'CONFIRMED']
false_positive = df[df['koi_disposition'] == 'FALSE POSITIVE']

confirmed = confirmed.sample(frac=1, random_state=42)
false_positive = false_positive.sample(frac=1, random_state=42)

confirmed_train = confirmed.iloc[:2000]
confirmed_test = confirmed.iloc[2000:]
false_pos_train = false_positive.iloc[:3500]
false_pos_test = false_positive.iloc[3500:]

train_df = pd.concat([confirmed_train, false_pos_train])
test_df = pd.concat([confirmed_test, false_pos_test])

drop_cols = ['koi_disposition', 'kepid', 'kepoi_name']
X_train = train_df.drop(columns=drop_cols).to_numpy()
X_test = test_df.drop(columns=drop_cols).to_numpy()
y_train = np.array([1]*len(confirmed_train) + [0]*len(false_pos_train))
y_test = np.array([1]*len(confirmed_test) + [0]*len(false_pos_test))

X_train = X_train.astype(float)
X_test = X_test.astype(float)

imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

conv_combinations = [
    (32,16),
    (64,32),
    (128,64),
    (16,32),
    (32,64),
    (64,128)
]

dense_combinations = [
    (32,16),
    (64,32),
    (128,64),
    (16,32),
    (32,64),
    (64,128)
]

activations = ["tanh", "relu", "leaky_relu", "elu"]

results = []

model_num = 1

for conv_filters in conv_combinations:
    for dense_units in dense_combinations:
        for activation in activations:
            print(f"\n=== Model {model_num}: Conv{conv_filters}, Dense{dense_units}, Activation={activation} ===")

            model = Sequential()
            if activation in ["tanh", "relu"]:
                model.add(Conv1D(conv_filters[0], kernel_size=3, activation=activation, input_shape=(X_train.shape[1],1)))
            else:
                model.add(Conv1D(conv_filters[0], kernel_size=3, input_shape=(X_train.shape[1],1)))
                if activation=="leaky_relu":
                    model.add(LeakyReLU())
                else:
                    model.add(ELU())
            if activation in ["tanh", "relu"]:
                model.add(Conv1D(conv_filters[1], kernel_size=3, activation=activation))
            else:
                model.add(Conv1D(conv_filters[1], kernel_size=3))
                if activation=="leaky_relu":
                    model.add(LeakyReLU())
                else:
                    model.add(ELU())
            model.add(Flatten())
            if activation in ["tanh", "relu"]:
                model.add(Dense(dense_units[0], activation=activation))
            else:
                model.add(Dense(dense_units[0]))
                if activation=="leaky_relu":
                    model.add(LeakyReLU())
                else:
                    model.add(ELU())
            if activation in ["tanh", "relu"]:
                model.add(Dense(dense_units[1], activation=activation))
            else:
                model.add(Dense(dense_units[1]))
                if activation=="leaky_relu":
                    model.add(LeakyReLU())
                else:
                    model.add(ELU())
            model.add(Dense(1, activation='sigmoid'))

            model.compile(optimizer=SGD(learning_rate=0.007), loss='binary_crossentropy', metrics=['accuracy'])
            model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

            y_pred = (model.predict(X_test) >= 0.5).astype(int).flatten()

            f1 = f1_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)

            results.append({
                "Model": model_num,
                "Conv1_filters": conv_filters[0],
                "Conv2_filters": conv_filters[1],
                "Dense1_units": dense_units[0],
                "Dense2_units": dense_units[1],
                "Activation": activation,
                "F1": f1,
                "Precision": precision,
                "Recall": recall
            })

            model_num +=1

results_df = pd.DataFrame(results)
results_df.to_csv("results.csv", index=False)

### **Classical Method Explanation**

This section implements a classical Convolutional Neural Network (CNN) to classify exoplanet candidates based on astronomical features. It involves data loading, preprocessing, model definition, training, and evaluation for various hyperparameter combinations.

#### **1. Library Imports**

The first part of the code imports necessary libraries for data manipulation, machine learning, and model building:

*   `pandas` as `pd`: For data handling and analysis, especially with DataFrames.
*   `numpy` as `np`: For numerical operations, particularly useful for array manipulations.
*   `sklearn.preprocessing.StandardScaler`: To standardize features by removing the mean and scaling to unit variance.
*   `sklearn.impute.SimpleImputer`: To handle missing values in the dataset.
*   `sklearn.metrics.f1_score`, `precision_score`, `recall_score`: For evaluating the performance of the classification model.
*   `tensorflow.keras.models.Sequential`: To build a linear stack of layers for the neural network.
*   `tensorflow.keras.layers.Dense`, `Conv1D`, `Flatten`, `LeakyReLU`, `ELU`: Various layers and activation functions for constructing the CNN.
*   `tensorflow.keras.optimizers.SGD`: The Stochastic Gradient Descent optimizer for training the model.

#### **2. Data Loading and Initial Preparation**

This segment loads the dataset and separates it into two classes: 'CONFIRMED' exoplanets and 'FALSE POSITIVE' detections.

*   `df = pd.read_excel("Final Data Compiled for BeQu.xlsx", sheet_name='FinalData')`: Loads data from an Excel file named 'Final Data Compiled for BeQu.xlsx' into a pandas DataFrame. It specifically reads the sheet named 'FinalData'.
*   `confirmed = df[df['koi_disposition'] == 'CONFIRMED']`: Filters the DataFrame `df` to select rows where the 'koi_disposition' column is 'CONFIRMED', representing confirmed exoplanets.
*   `false_positive = df[df['koi_disposition'] == 'FALSE POSITIVE']`: Filters the DataFrame `df` to select rows where the 'koi_disposition' column is 'FALSE POSITIVE', representing non-exoplanets.

#### **3. Data Sampling and Splitting**

The 'CONFIRMED' and 'FALSE POSITIVE' datasets are shuffled and then split into training and testing sets.

*   `confirmed = confirmed.sample(frac=1, random_state=42)`: Shuffles the 'confirmed' DataFrame randomly. `frac=1` means all rows are included in the sample, and `random_state=42` ensures reproducibility.
*   `false_positive = false_positive.sample(frac=1, random_state=42)`: Shuffles the 'false_positive' DataFrame similarly.
*   `confirmed_train = confirmed.iloc[:2000]`: Takes the first 2000 rows of the shuffled 'confirmed' data for the training set.
*   `confirmed_test = confirmed.iloc[2000:]`: Takes the remaining rows of the shuffled 'confirmed' data for the testing set.
*   `false_pos_train = false_positive.iloc[:3500]`: Takes the first 3500 rows of the shuffled 'false_positive' data for the training set.
*   `false_pos_test = false_positive.iloc[3500:]`: Takes the remaining rows of the shuffled 'false_positive' data for the testing set.
*   `train_df = pd.concat([confirmed_train, false_pos_train])`: Combines the 'confirmed' and 'false positive' training sets into a single DataFrame for training.
*   `test_df = pd.concat([confirmed_test, false_pos_test])`: Combines the 'confirmed' and 'false positive' testing sets into a single DataFrame for testing.

#### **4. Feature and Label Separation**

This part separates the features (X) from the target labels (y) and prepares them for the model.

*   `drop_cols = ['koi_disposition', 'kepid', 'kepoi_name']`: Defines a list of columns to be dropped from the feature set. 'koi_disposition' is the target variable, and 'kepid' and 'kepoi_name' are identifiers not relevant for training.
*   `X_train = train_df.drop(columns=drop_cols).to_numpy()`: Creates the training feature set `X_train` by dropping the specified columns from `train_df` and converting it to a NumPy array.
*   `X_test = test_df.drop(columns=drop_cols).to_numpy()`: Creates the testing feature set `X_test` similarly from `test_df`.
*   `y_train = np.array([1]*len(confirmed_train) + [0]*len(false_pos_train))`: Creates the training labels `y_train`. 'CONFIRMED' examples are labeled as 1, and 'FALSE POSITIVE' examples are labeled as 0.
*   `y_test = np.array([1]*len(confirmed_test) + [0]*len(false_pos_test))`: Creates the testing labels `y_test` using the same logic.

#### **5. Data Type Conversion, Imputation, and Scaling**

Before feeding the data into the neural network, numerical preprocessing steps are applied.

*   `X_train = X_train.astype(float)` and `X_test = X_test.astype(float)`: Ensures that the feature arrays are of float type, which is typically required for numerical computations in machine learning models.
*   `imputer = SimpleImputer(strategy='mean')`: Initializes a `SimpleImputer` to fill missing values (NaNs) with the mean of each column.
*   `X_train = imputer.fit_transform(X_train)`: Fits the imputer on the training data and then transforms `X_train` by filling missing values.
*   `X_test = imputer.transform(X_test)`: Transforms the testing data `X_test` using the imputer fitted on the training data (it's crucial to avoid data leakage from the test set).
*   `scaler = StandardScaler()`: Initializes a `StandardScaler` to standardize features.
*   `X_train = scaler.fit_transform(X_train)`: Fits the scaler on the training data and then transforms `X_train`.
*   `X_test = scaler.transform(X_test)`: Transforms `X_test` using the scaler fitted on the training data.

#### **6. Reshaping Data for Conv1D Layer**

Convolutional layers (like `Conv1D`) expect a specific input shape, which is handled here.

*   `X_train = X_train[..., np.newaxis]`: Adds a new dimension to `X_train` to make its shape `(samples, timesteps, features)`. For `Conv1D`, `features` is typically 1 when each feature is treated as a separate channel or time-step for the convolution.
*   `X_test = X_test[..., np.newaxis]`: Reshapes `X_test` in the same way.

#### **7. Hyperparameter Combinations**

This section defines lists of hyperparameters to be systematically tested during the model training and evaluation process.

*   `conv_combinations`: A list of tuples, each representing a pair of filter sizes for the two `Conv1D` layers in the network (e.g., `(32, 16)` means the first `Conv1D` layer will have 32 filters and the second will have 16 filters).
*   `dense_combinations`: A list of tuples, each representing a pair of unit counts for the two `Dense` (fully connected) layers.
*   `activations`: A list of different activation functions (`'tanh'`, `'relu'`, `'leaky_relu'`, `'elu'`) to be experimented with.

#### **8. Model Training and Evaluation Loop**

This is the core of the classical method, where different CNN configurations are built, trained, and evaluated in a nested loop.

*   `results = []`: An empty list to store the performance metrics of each model configuration.
*   `model_num = 1`: Initializes a counter for the models.
*   **Nested Loops**: The code iterates through every combination of `conv_filters`, `dense_units`, and `activation` from the defined lists. For each combination:
    *   `print(f"\n=== Model {model_num}: Conv{conv_filters}, Dense{dense_units}, Activation={activation} ===")`: Prints the current model's configuration.
    *   `model = Sequential()`: Initializes a new Keras Sequential model for each iteration.
    *   **Conditional Layer Addition**: The `Conv1D` and `Dense` layers are added to the model. The activation function is applied conditionally:
        *   For `'tanh'` and `'relu'`, the activation is directly specified in the layer constructor.
        *   For `'leaky_relu'` and `'elu'`, these are added as separate layers after the `Conv1D` or `Dense` layer, as they are not directly supported as activation strings in the layer constructors in this specific way (though `LeakyReLU` can be passed as an `Activation` layer).
    *   `model.add(Conv1D(...))`: Adds two `Conv1D` layers. The first `Conv1D` layer requires `input_shape`.
    *   `model.add(Flatten())`: Flattens the output of the convolutional layers into a 1D vector to be fed into the dense layers.
    *   `model.add(Dense(...))`: Adds two `Dense` layers.
    *   `model.add(Dense(1, activation='sigmoid'))`: Adds the final output layer with a single neuron and a 'sigmoid' activation function, suitable for binary classification (output between 0 and 1).
    *   `model.compile(optimizer=SGD(learning_rate=0.007), loss='binary_crossentropy', metrics=['accuracy'])`: Configures the model for training:
        *   `optimizer=SGD(learning_rate=0.007)`: Uses Stochastic Gradient Descent with a specified learning rate.
        *   `loss='binary_crossentropy'`: The loss function for binary classification.
        *   `metrics=['accuracy']`: Tracks accuracy during training.
    *   `model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)`: Trains the model on the `X_train` and `y_train` data for 20 epochs with a batch size of 32. `verbose=0` means no output during training.
    *   `y_pred = (model.predict(X_test) >= 0.5).astype(int).flatten()`: Makes predictions on the test set. The `model.predict()` output (probabilities) is converted into binary predictions (0 or 1) using a threshold of 0.5, and then flattened.
    *   `f1 = f1_score(y_test, y_pred)`, `precision = precision_score(y_test, y_pred)`, `recall = recall_score(y_test, y_pred)`: Calculates the F1-score, Precision, and Recall using the true labels (`y_test`) and predicted labels (`y_pred`).
    *   `results.append(...)`: Stores all the model's configuration details and evaluation metrics in the `results` list.
    *   `model_num += 1`: Increments the model counter.

#### **9. Results Storage**

Finally, the accumulated results are saved to a CSV file.

*   `results_df = pd.DataFrame(results)`: Converts the `results` list (containing dictionaries for each model) into a pandas DataFrame.
*   `results_df.to_csv("results.csv", index=False)`: Saves the `results_df` DataFrame to a CSV file named 'results.csv'. `index=False` prevents writing the DataFrame index as a column in the CSV.

In [ ]:
#Quantum Model

!pip uninstall -y jax jaxlib
!pip install "jax==0.4.28" "jaxlib==0.4.28"
!pip install pennylane
!pip install pennylane-lightning
!pip uninstall flax orbax-checkpoint -y

import pennylane as qml
from pennylane import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

df = pd.read_excel("Final Data Compiled for BeQu.xlsx", sheet_name='FinalData')

confirmed = df[df['koi_disposition'] == 'CONFIRMED']
false_positive = df[df['koi_disposition'] == 'FALSE POSITIVE']

confirmed = confirmed.sample(frac=1, random_state=42)
false_positive = false_positive.sample(frac=1, random_state=42)

confirmed_train = confirmed.iloc[:2000]
confirmed_test = confirmed.iloc[2000:]
false_pos_train = false_positive.iloc[:2000]
false_pos_test = false_positive.iloc[3500:]

train_df = pd.concat([confirmed_train, false_pos_train])
test_df = pd.concat([confirmed_test, false_pos_test])

drop_cols = ['koi_disposition', 'kepid', 'kepoi_name']
X_train = train_df.drop(columns=drop_cols).to_numpy()
X_test = test_df.drop(columns=drop_cols).to_numpy()
y_train = np.array([1]*len(confirmed_train) + [0]*len(false_pos_train))
y_test = np.array([1]*len(confirmed_test) + [0]*len(false_pos_test))

imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

n_qubits = 4
features_per_qubit = 3

assert X_train.shape[1] == n_qubits * features_per_qubit, "Feature count must match qubits × rotations"

dev = qml.device("lightning.qubit", wires=n_qubits)

def angle_encoding(x):
    for i in range(n_qubits):
        qml.RY(x[i*3], wires=i)
        qml.RZ(x[i*3 + 1], wires=i)
        qml.RX(x[i*3 + 2], wires=i)

n_layers = 2 #Speed Up Step
def ansatz(params):
    qml.templates.BasicEntanglerLayers(weights=params, wires=range(n_qubits))

@qml.qnode(dev)
def circuit(x, params):
    angle_encoding(x)
    ansatz(params)
    return qml.expval(qml.PauliZ(0))

def predict_probs(X, params):
    return np.array([circuit(x, params) for x in X])

def loss(params, X, y):
    preds = predict_probs(X, params)
    preds = 0.5 * (1 - preds)
    return -np.mean(y * np.log(preds + 1e-8) + (1 - y) * np.log(1 - preds + 1e-8))

from scipy.optimize import minimize

np.random.seed(42)
init_params = np.random.uniform(0, 2*np.pi, size=(n_layers, n_qubits))

opt_result = minimize(
    fun=lambda p: loss(p.reshape(n_layers, n_qubits), X_train, y_train),
    x0=init_params.ravel(),
    method="COBYLA",
    options={"maxiter":250, "disp":True} #Speed Up Step
)

trained_params = opt_result.x.reshape(n_layers, n_qubits)

y_pred_probs = predict_probs(X_test, trained_params)
y_pred_probs = 0.5 * (1 - y_pred_probs)
y_pred = (y_pred_probs >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(report)